# 02 Structured Multi-Tenant RLS with Row-Level Security

## Overview

In this tutorial we'll explore a different approach to multi-tenancy - using Row-Level Security (RLS) in PostgreSQL for structured data. Unlike Tutorial 01 that used Knowledge Bases for unstructured documents, this tutorial focuses on querying structured data like insurance claims from a relational database.

We use Access Tokens with a database-level isolation mechanism. The agent extracts the user's `sub` claim from the JWT, looks up their tenant in a mapping table, and sets the PostgreSQL session context for RLS filtering.

In this example, we'll:
1. Set up a PostgreSQL database with Row-Level Security
2. Create a Cognito User Pool with Access Token authentication
3. Build a user-to-tenant mapping system
4. Create a Strands Agent that queries the database with RLS
5. Deploy to AgentCore Runtime and test multi-tenant isolation

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Structured Data with RLS |
| Authentication | OAuth 2.0 Access Token |
| Agentic Framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet 4.5 |
| Data store | PostgreSQL with RLS |
| Tenant isolation | Row-Level Security (database layer) |
| Tutorial vertical | Insurance Claims |
| Example complexity | Intermediate |
| SDK used | Amazon Bedrock AgentCore Python SDK |

### Tutorial Architecture

```
Insurance Broker → Access Token → AgentCore → Strands Agent
                   (OAuth 2.0)                      ↓
                                        Extract sub from JWT
                                                     ↓
                                        Lookup tenant from user_tenants table
                                                     ↓
                                        SET app.current_tenant_id = 'tenant-a'
                                                     ↓
                                        PostgreSQL Query + RLS Policy
           ↓
                                        Returns only tenant's claims
```

## Prerequisites

To execute this tutorial you will need:
* Python 3.9+
* AWS credentials configured
* Amazon Bedrock access
* Bedrock AgentCore SDK
* Strands Agents
* **PostgreSQL database** (Aurora Serverless v2 - created automatically)

In [ ]:
!pip install -qU boto3 bedrock-agentcore bedrock-agentcore-starter-toolkit strands-agents psycopg2-binary pyyaml

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import boto3
import json
import time
import yaml
import uuid
import importlib
from boto3.session import Session

# Reload utils module to ensure latest code is used
from utils import aws_resources
importlib.reload(aws_resources)
from utils.aws_resources import CognitoManager, AuroraServerlessManager

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print("\n✓ AWS utilities loaded")

## Configuration

In [ ]:
PROJECT_NAME = "structured-rls"
user_pool_name = f"{PROJECT_NAME}-pool"
cluster_name = f"{PROJECT_NAME}-aurora"

print("Configuration:")
print(f"  User Pool: {user_pool_name}")
print(f"  Aurora Cluster: {cluster_name}")

## Setting Up Cognito with Access Tokens

In [ ]:
cognito = CognitoManager(region)
cognito_client = boto3.client('cognito-idp', region_name=region)

cognito_config = cognito.get_or_create_user_pool(
    pool_name=user_pool_name,
    schema=[
        {'Name': 'email', 'AttributeDataType': 'String', 'Mutable': True, 'Required': True}
    ],
    auto_verified_attributes=['email']
)

user_pool_id = cognito_config['user_pool_id']
client_id = cognito_config['client_id']
discovery_url = cognito_config['discovery_url']

print(f"✓ User Pool ID: {user_pool_id}")
print(f"✓ Client ID: {client_id}")

### Creating Test Users

We create test users for different insurance brokers. The tenant association is stored in the database's `user_tenants` table.

In [ ]:
test_users = [
    {
        'username': 'mike-broker-a',
        'password': 'Pass123!',
        'email': 'mike.broker@example.com',
        'tenant_id': 'broker-a'
    },
    {
        'username': 'emma-broker-b',
        'password': 'Pass123!',
        'email': 'emma.broker@example.com',
        'tenant_id': 'broker-b'
    }
]

# Store user subs for later tenant mapping
user_subs = {}

for user in test_users:
    result = cognito.get_or_create_user(
        user_pool_id=user_pool_id,
        username=user['username'],
        password=user['password'],
        attributes=[
            {'Name': 'email', 'Value': user['email']},
            {'Name': 'email_verified', 'Value': 'true'}
        ]
    )
    user_subs[user['username']] = result.get('sub')
    print(f"  Created user: {user['username']} (sub: {result.get('sub')})")

print(f"\n✓ {len(test_users)} users ready")

## Setting Up Aurora Serverless v2 with Row-Level Security

We'll create an Aurora Serverless v2 PostgreSQL cluster. Aurora Serverless v2 automatically scales capacity based on demand and can scale down to save costs when idle. RLS automatically filters query results based on the current user context.

In [ ]:
# Create Aurora Serverless v2 cluster
aurora = AuroraServerlessManager(region)
aurora_config = aurora.get_or_create_cluster(
    cluster_name=cluster_name,
    database_name='insurance_claims',
    master_username='postgres'
)

DB_HOST = aurora_config['endpoint']
DB_NAME = aurora_config['database']
DB_USER = aurora_config.get('master_username', 'postgres')
DB_PASSWORD = aurora_config.get('master_password')
DB_PORT = aurora_config['port']
CLUSTER_ARN = aurora_config['cluster_arn']
SECRET_ARN = aurora_config['secret_arn']

# Initialize RDS Data API client
rds_data = boto3.client('rds-data', region_name=region)

print(f"\n✓ Aurora Serverless v2 ready")
print(f"  Endpoint: {DB_HOST}")
print(f"  Cluster ARN: {CLUSTER_ARN}")

In [ ]:
# Wait for DB instance to be available (Data API requires instance to be ready)
rds_client = boto3.client('rds', region_name=region)
instance_id = f"{cluster_name}-instance"

print(f"Checking instance status: {instance_id}")
while True:
    try:
        response = rds_client.describe_db_instances(DBInstanceIdentifier=instance_id)
        status = response['DBInstances'][0]['DBInstanceStatus']
        if status == 'available':
            print(f"✓ Instance is available")
            break
        else:
            print(f"  ⏳ Instance status: {status}, waiting...")
            time.sleep(30)
    except Exception as e:
        print(f"  ⏳ Waiting for instance... ({e})")
        time.sleep(30)

# Helper function to execute SQL via Data API
def execute_sql(sql, parameters=None):
    """Execute SQL statement using RDS Data API"""
    params = {
        'resourceArn': CLUSTER_ARN,
        'secretArn': SECRET_ARN,
        'database': DB_NAME,
        'sql': sql
    }
    if parameters:
        params['parameters'] = parameters
    return rds_data.execute_statement(**params)

# Test connection via Data API
try:
    result = execute_sql('SELECT 1 as test')
    print(f"✓ Connected to Aurora via Data API")
    print(f"  Database: {DB_NAME}")
except Exception as e:
    print(f"✗ Connection failed: {e}")

In [ ]:
# Create database schema using Data API
# Create tenants table
execute_sql("""
    CREATE TABLE IF NOT EXISTS tenants (
        tenant_id VARCHAR(50) PRIMARY KEY,
        name VARCHAR(100) NOT NULL
    )
""")
print("  ✓ Created tenants table")

# Create user_tenants mapping table
execute_sql("""
    CREATE TABLE IF NOT EXISTS user_tenants (
        user_sub VARCHAR(100) PRIMARY KEY,
        tenant_id VARCHAR(50) REFERENCES tenants(tenant_id)
    )
""")
print("  ✓ Created user_tenants table")

# Create claims table with RLS
execute_sql("""
    CREATE TABLE IF NOT EXISTS claims (
        id SERIAL PRIMARY KEY,
        tenant_id VARCHAR(50) REFERENCES tenants(tenant_id),
        claim_number VARCHAR(20) NOT NULL,
        customer_name VARCHAR(100) NOT NULL,
        claim_amount DECIMAL(10,2) NOT NULL,
        claim_date DATE NOT NULL,
        status VARCHAR(20) NOT NULL,
        description TEXT
    )
""")
print("  ✓ Created claims table")

# Enable RLS on claims table
execute_sql("ALTER TABLE claims ENABLE ROW LEVEL SECURITY")
print("  ✓ Enabled RLS on claims table")

# Create RLS policy
execute_sql("DROP POLICY IF EXISTS tenant_isolation ON claims")
execute_sql("""
    CREATE POLICY tenant_isolation ON claims
        FOR ALL
        USING (tenant_id = current_setting('app.current_tenant_id', true))
""")
print("  ✓ Created RLS policy")

# Create app user for RLS enforcement (superuser bypasses RLS)
APP_DB_USER = 'app_user'
APP_DB_PASSWORD = DB_PASSWORD  # Reuse same password for simplicity

execute_sql(f"DROP ROLE IF EXISTS {APP_DB_USER}")
execute_sql(f"CREATE ROLE {APP_DB_USER} WITH LOGIN PASSWORD '{APP_DB_PASSWORD}'")
execute_sql(f"GRANT CONNECT ON DATABASE {DB_NAME} TO {APP_DB_USER}")
execute_sql(f"GRANT USAGE ON SCHEMA public TO {APP_DB_USER}")
execute_sql(f"GRANT SELECT ON ALL TABLES IN SCHEMA public TO {APP_DB_USER}")
print("  ✓ Created app_user for RLS enforcement")

print("\n✓ Database schema created with RLS enabled")

In [ ]:
# Insert sample data using Data API
# Insert tenants
execute_sql("""
    INSERT INTO tenants (tenant_id, name) VALUES
    ('broker-a', 'Broker A Insurance'),
    ('broker-b', 'Broker B Insurance')
    ON CONFLICT (tenant_id) DO NOTHING
""")
print("  ✓ Inserted tenants")

# Insert claims for Broker A
execute_sql("""
    INSERT INTO claims (tenant_id, claim_number, customer_name, claim_amount, claim_date, status) VALUES
    ('broker-a', 'CLM-A-001', 'John Doe', 5000.00, CURRENT_DATE - INTERVAL '2 days', 'approved'),
    ('broker-a', 'CLM-A-002', 'Jane Smith', 3500.00, CURRENT_DATE - INTERVAL '5 days', 'pending'),
    ('broker-a', 'CLM-A-003', 'Bob Johnson', 7200.00, CURRENT_DATE - INTERVAL '1 day', 'approved')
    ON CONFLICT DO NOTHING
""")
print("  ✓ Inserted Broker A claims")

# Insert claims for Broker B
execute_sql("""
    INSERT INTO claims (tenant_id, claim_number, customer_name, claim_amount, claim_date, status) VALUES
    ('broker-b', 'CLM-B-001', 'David Lee', 6500.00, CURRENT_DATE - INTERVAL '4 days', 'approved'),
    ('broker-b', 'CLM-B-002', 'Emma Davis', 4800.00, CURRENT_DATE - INTERVAL '2 days', 'pending')
    ON CONFLICT DO NOTHING
""")
print("  ✓ Inserted Broker B claims")

print("\n✓ Sample data inserted")

In [ ]:
# Insert user-tenant mappings using the user subs from Cognito
for user in test_users:
    user_sub = user_subs.get(user['username'])
    if user_sub:
        execute_sql(
            "INSERT INTO user_tenants (user_sub, tenant_id) VALUES (:sub, :tenant) ON CONFLICT (user_sub) DO UPDATE SET tenant_id = EXCLUDED.tenant_id",
            parameters=[
                {'name': 'sub', 'value': {'stringValue': user_sub}},
                {'name': 'tenant', 'value': {'stringValue': user['tenant_id']}}
            ]
        )
        print(f"  Mapped {user['username']} (sub: {user_sub[:8]}...) → {user['tenant_id']}")

print("\n✓ User-tenant mappings inserted")

### Testing Row-Level Security

In [ ]:
# Test RLS using Data API (note: Data API doesn't support SET in same transaction easily)
# We'll verify data was inserted correctly by querying without RLS filter
def query_all_claims():
    result = execute_sql("SELECT tenant_id, claim_number, customer_name, claim_amount, status FROM claims ORDER BY tenant_id, claim_number")
    return result.get('records', [])

print("=== All Claims (without RLS filter) ===")
for record in query_all_claims():
    tenant = record[0].get('stringValue', '')
    claim_num = record[1].get('stringValue', '')
    customer = record[2].get('stringValue', '')
    amount = record[3].get('stringValue', record[3].get('doubleValue', ''))
    print(f"  [{tenant}] {claim_num}: {customer} - ${amount}")

print("\n✓ Data inserted successfully")
print("  Note: RLS will be enforced when agent queries via psycopg2 in VPC")

## Creating the Multi-Tenant RLS Agent

The agent extracts the user's `sub` from the JWT, looks up their tenant, and sets the PostgreSQL session context for RLS filtering.

In [ ]:
%%writefile structured_multitenant_rls_agent.py

from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore import BedrockAgentCoreApp
from bedrock_agentcore.runtime import BedrockAgentCoreContext
import json
import os
import base64
import psycopg2
from psycopg2.extras import RealDictCursor

app = BedrockAgentCoreApp()
model = BedrockModel(model_id="global.anthropic.claude-sonnet-4-5-20250929-v1:0")

DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME", "insurance_claims")
DB_USER = os.getenv("DB_USER", "app_user")  # Use app_user for RLS enforcement
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_PORT = os.getenv("DB_PORT", "5432")


def decode_jwt_payload(token: str) -> dict:
    try:
        parts = token.split('.')
        if len(parts) != 3:
            return {}
        payload_part = parts[1]
        padding = 4 - len(payload_part) % 4
        if padding != 4:
            payload_part += '=' * padding
        return json.loads(base64.urlsafe_b64decode(payload_part))
    except Exception as e:
        print(f"JWT decode error: {e}")
        return {}


def extract_user_from_token():
    jwt_claims = {}
    try:
        headers = BedrockAgentCoreContext.get_request_headers()
        auth_header = headers.get('Authorization', '') if headers else ''
        if auth_header.startswith('Bearer '):
            jwt_claims = decode_jwt_payload(auth_header[7:])
    except Exception as e:
        print(f"Error extracting user: {e}")
    return jwt_claims


def get_db_connection():
    return psycopg2.connect(
        host=DB_HOST, database=DB_NAME, user=DB_USER,
        password=DB_PASSWORD, port=DB_PORT
    )


def get_tenant_for_user(user_sub: str) -> str:
    conn = get_db_connection()
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT tenant_id FROM user_tenants WHERE user_sub = %s", (user_sub,))
            result = cur.fetchone()
            return result[0] if result else None
    finally:
        conn.close()


def create_query_claims_tool(tenant_id: str):
    """Create a query_claims tool with tenant_id bound via closure."""
    @tool
    def query_claims() -> str:
        """Query insurance claims for the current user. Returns only claims the user is authorized to see."""
        # SECURITY: tenant_id is bound at tool creation time, not from LLM input
        conn = get_db_connection()
        try:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("SET app.current_tenant_id = %s", (tenant_id,))
                cur.execute("SELECT claim_number, customer_name, claim_amount, status FROM claims")
                results = cur.fetchall()
                return json.dumps({"claims": [dict(r) for r in results], "tenant_id": tenant_id}, default=str)
        finally:
            conn.close()
    return query_claims


@app.entrypoint
def invoke_agent(payload):
    user_input = payload.get("prompt", "")
    jwt_claims = extract_user_from_token()
    user_sub = jwt_claims.get("sub")
    username = jwt_claims.get("username", jwt_claims.get("cognito:username", "unknown"))
    
    tenant_id = get_tenant_for_user(user_sub) if user_sub else None
    
    if not tenant_id:
        return {"response": "Tenant not found.", "tenant_found": False}
    
    # SECURITY: Create tool with tenant_id bound - LLM cannot override
    query_claims_tool = create_query_claims_tool(tenant_id)
    
    # Create agent per-request with bound tool
    agent = Agent(
        model=model,
        system_prompt="""You are an insurance claims assistant. Use query_claims to get claims data.
IMPORTANT: You can only access claims for the authenticated user's tenant. 
Do not attempt to query other tenants - the system enforces row-level security.""",
        tools=[query_claims_tool]
    )
    
    response = agent(user_input)
    
    return {
        "response": response.message["content"][0]["text"],
        "user_identity": {"sub": user_sub, "username": username, "tenant_id": tenant_id},
        "rls_applied": True
    }


if __name__ == "__main__":
    app.run()

### Deploying to AgentCore Runtime

#### Setting Up VPC Endpoints

When running AgentCore in VPC mode without internet access (no NAT Gateway), we need VPC Endpoints for the agent to communicate with AWS services like Bedrock.

In [ ]:
# Create VPC Endpoints for AgentCore to access AWS services
ec2 = boto3.client('ec2', region_name=region)

# Get default VPC
vpcs = ec2.describe_vpcs(Filters=[{'Name': 'is-default', 'Values': ['true']}])
vpc_id = vpcs['Vpcs'][0]['VpcId']

# Get Aurora security group (will be used for endpoints too)
aurora_sg_name = f"{cluster_name}-sg"
sgs = ec2.describe_security_groups(Filters=[{'Name': 'group-name', 'Values': [aurora_sg_name]}])
sg_id = sgs['SecurityGroups'][0]['GroupId']

# Supported AZ IDs per region for AgentCore VPC
AGENTCORE_SUPPORTED_AZS = {
    'us-east-1': ['use1-az1', 'use1-az2', 'use1-az4'],
    'us-east-2': ['use2-az1', 'use2-az2', 'use2-az3'],
    'us-west-2': ['usw2-az1', 'usw2-az2', 'usw2-az3'],
    'ap-southeast-2': ['apse2-az1', 'apse2-az2', 'apse2-az3'],
    'ap-south-1': ['aps1-az1', 'aps1-az2', 'aps1-az3'],
    'ap-southeast-1': ['apse1-az1', 'apse1-az2', 'apse1-az3'],
    'ap-northeast-1': ['apne1-az1', 'apne1-az2', 'apne1-az4'],
    'eu-west-1': ['euw1-az1', 'euw1-az2', 'euw1-az3'],
    'eu-central-1': ['euc1-az1', 'euc1-az2', 'euc1-az3'],
}

# Get subnets in supported AZs
supported_az_ids = AGENTCORE_SUPPORTED_AZS.get(region, [])
subnets = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}])
subnet_ids = [
    s['SubnetId'] for s in subnets['Subnets'] 
    if s.get('AvailabilityZoneId') in supported_az_ids
][:2]

# Track created endpoints for cleanup
created_vpc_endpoints = []

# Add HTTPS inbound rule (self-referencing) for VPC Endpoint communication
# Agent and VPC Endpoint ENIs share the same SG, so agent needs to reach endpoints on 443
try:
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[{
            'IpProtocol': 'tcp',
            'FromPort': 443,
            'ToPort': 443,
            'UserIdGroupPairs': [{'GroupId': sg_id}]
        }]
    )
    print(f"✓ Added HTTPS inbound rule (self-referencing) to {sg_id}")
except Exception as e:
    if 'Duplicate' in str(e):
        print("↻ HTTPS inbound rule already exists")
    else:
        print(f"SG rule error: {e}")

def create_interface_endpoint(service_name, display_name):
    try:
        response = ec2.create_vpc_endpoint(
            VpcId=vpc_id,
            ServiceName=f'com.amazonaws.{region}.{service_name}',
            VpcEndpointType='Interface',
            SubnetIds=subnet_ids,
            SecurityGroupIds=[sg_id],
            PrivateDnsEnabled=True
        )
        created_vpc_endpoints.append(response['VpcEndpoint']['VpcEndpointId'])
        print(f"✓ Created {display_name} endpoint: {response['VpcEndpoint']['VpcEndpointId']}")
    except Exception as e:
        if 'already exists' in str(e).lower() or 'duplicate' in str(e).lower():
            print(f"↻ {display_name} endpoint already exists")
        else:
            print(f"{display_name} error: {e}")

# 1. AgentCore Data Plane Endpoint (required for health checks)
create_interface_endpoint('bedrock-agentcore', 'bedrock-agentcore')

# 2. AgentCore Gateway Endpoint (required for VPC mode)
create_interface_endpoint('bedrock-agentcore.gateway', 'bedrock-agentcore-gateway')

# 3. Bedrock Runtime Endpoint (required for model invocation)
create_interface_endpoint('bedrock-runtime', 'bedrock-runtime')

# 4. ECR API Endpoint (required for container pull)
create_interface_endpoint('ecr.api', 'ecr-api')

# 5. ECR DKR Endpoint (required for container pull)
create_interface_endpoint('ecr.dkr', 'ecr-dkr')

# 6. S3 Gateway Endpoint (required for ECR layer storage)
try:
    rts = ec2.describe_route_tables(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}])
    rt_ids = [rt['RouteTableId'] for rt in rts['RouteTables']]
    
    response = ec2.create_vpc_endpoint(
        VpcId=vpc_id,
        ServiceName=f'com.amazonaws.{region}.s3',
        VpcEndpointType='Gateway',
        RouteTableIds=rt_ids
    )
    created_vpc_endpoints.append(response['VpcEndpoint']['VpcEndpointId'])
    print(f"✓ Created S3 gateway endpoint: {response['VpcEndpoint']['VpcEndpointId']}")
except Exception as e:
    if 'already exists' in str(e).lower() or 'duplicate' in str(e).lower():
        print("↻ S3 endpoint already exists")
    else:
        print(f"S3 error: {e}")

# 7. CloudWatch Logs Endpoint
create_interface_endpoint('logs', 'logs')

# 8. CloudWatch Monitoring Endpoint (for telemetry/metrics)
create_interface_endpoint('monitoring', 'monitoring')

# Wait for endpoints to be available
if created_vpc_endpoints:
    print("\n⏳ Waiting for VPC endpoints to be available...")
    time.sleep(60)
    print("✓ VPC endpoints ready")
else:
    print("\n✓ All VPC endpoints already exist")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# VPC config from previous cell
print(f"VPC Config:")
print(f"  VPC: {vpc_id}")
print(f"  Subnets: {subnet_ids}")
print(f"  Security Group: {sg_id}")

runtime = Runtime()

runtime.configure(
    entrypoint="structured_multitenant_rls_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name="structured_multitenant_rls",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    },
    request_header_configuration={
        'requestHeaderAllowlist': ['Authorization']
    },
    vpc_enabled=True,
    vpc_subnets=subnet_ids,
    vpc_security_groups=[sg_id]
)

print("\n✓ Agent configured with VPC mode")

In [ ]:
print("Deploying agent...\n")

# Pass DB credentials as environment variables
db_env_vars = {
    'DB_HOST': DB_HOST,
    'DB_NAME': DB_NAME,
    'DB_USER': 'app_user',  # Use app_user for RLS enforcement (not postgres superuser)
    'DB_PASSWORD': DB_PASSWORD,
    'DB_PORT': str(DB_PORT),
    'OTEL_SDK_DISABLED': 'true'  # Disable OTEL to avoid timeout in VPC mode
}

launch_result = runtime.launch(auto_update_on_conflict=True, env_vars=db_env_vars)
print(f"\n✓ Deployed: {launch_result.agent_arn}")

In [ ]:
print("Waiting for agent...")
status_response = runtime.status()
status = status_response.endpoint['status']

while status not in ['READY', 'CREATE_FAILED']:
    time.sleep(10)
    status_response = runtime.status()
    status = status_response.endpoint['status']

print(f"✓ Agent ready: {status}")

## Testing Multi-Tenant Isolation

In [ ]:
def get_tokens(username, password):
    response = cognito_client.initiate_auth(
        ClientId=client_id,
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password}
    )
    return response['AuthenticationResult']

print("Authenticating users...")
broker_a_tokens = get_tokens('mike-broker-a', 'Pass123!')
broker_b_tokens = get_tokens('emma-broker-b', 'Pass123!')
print("✓ Users authenticated")

In [ ]:
print("💼 TEST: Broker A - Claims Query")
print("="*60)

session_id_a = f"broker-a-session-{uuid.uuid4().hex}"

response_a = runtime.invoke(
    {"prompt": "Show me all my claims"},
    bearer_token=broker_a_tokens['AccessToken'],
    session_id=session_id_a
)

resp_data = response_a.get('response')
if isinstance(resp_data, str):
    try:
        parsed = json.loads(resp_data)
    except:
        parsed = {'response': resp_data}
else:
    parsed = resp_data

print(f"\nResponse: {parsed.get('response', 'N/A')}")
print(f"\nRLS Applied: {parsed.get('rls_applied', False)}")

In [ ]:
print("\n💎 TEST: Broker B - Claims Query")
print("="*60)

session_id_b = f"broker-b-session-{uuid.uuid4().hex}"

response_b = runtime.invoke(
    {"prompt": "Show me all my claims"},
    bearer_token=broker_b_tokens['AccessToken'],
    session_id=session_id_b
)

resp_data = response_b.get('response')
if isinstance(resp_data, str):
    try:
        parsed = json.loads(resp_data)
    except:
        parsed = {'response': resp_data}
else:
    parsed = resp_data

print(f"\nResponse: {parsed.get('response', 'N/A')}")
print(f"\nRLS Applied: {parsed.get('rls_applied', False)}")

print("\n" + "="*60)
print("✓ Multi-tenant RLS isolation verified!")

## Cleanup (Optional)

Uncomment and run to delete all resources.

In [ ]:
# from utils.aws_resources import cleanup_cognito_resources, cleanup_aurora_resources
# import subprocess

# # Cleanup AgentCore Runtime
# try:
#     print("Destroying AgentCore Runtime...")
#     result = subprocess.run(['agentcore', 'destroy', '--force'], capture_output=True, text=True, timeout=120)
#     print("✓ AgentCore Runtime destroyed")
# except Exception as e:
#     print(f"⚠ AgentCore cleanup: {e}")

# # Cleanup VPC Endpoints
# try:
#     print("\nDeleting VPC Endpoints...")
#     ec2 = boto3.client('ec2', region_name=region)
#     vpcs = ec2.describe_vpcs(Filters=[{'Name': 'is-default', 'Values': ['true']}])
#     vpc_id = vpcs['Vpcs'][0]['VpcId']
    
#     # Find endpoints created by this tutorial
#     endpoints = ec2.describe_vpc_endpoints(
#         Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]
#     )
    
#     endpoint_ids = []
#     for ep in endpoints.get('VpcEndpoints', []):
#         service = ep.get('ServiceName', '')
#         if any(svc in service for svc in ['bedrock-runtime', 'bedrock-agentcore', 'logs', 'ecr.api', 'ecr.dkr', 'monitoring']):
#             endpoint_ids.append(ep['VpcEndpointId'])
#             print(f"  Found: {ep['VpcEndpointId']} ({service.split('.')[-1]})")
#         elif 's3' in service and ep.get('VpcEndpointType') == 'Gateway':
#             endpoint_ids.append(ep['VpcEndpointId'])
#             print(f"  Found: {ep['VpcEndpointId']} (s3-gateway)")
    
#     if endpoint_ids:
#         ec2.delete_vpc_endpoints(VpcEndpointIds=endpoint_ids)
#         print(f"✓ Deleted {len(endpoint_ids)} VPC endpoints")
#     else:
#         print("  No VPC endpoints to delete")
# except Exception as e:
#     print(f"⚠ VPC Endpoints cleanup: {e}")

# # Cleanup Aurora
# cleanup_aurora_resources(region=region, cluster_name=cluster_name)

# # Cleanup Cognito
# cleanup_cognito_resources(region=region, user_pool_name=user_pool_name)

# Congratulations!

You've successfully built a multi-tenant system with Row-Level Security using Aurora Serverless v2!